# Validação e análise territorial — população IBGE municipal

**Objetivo:** auditar a tabela municipal de população para 2024 e 2025, validar códigos e somas contra os totais por UF e identificar municípios com maior população. A tabela é adequada para preparar denominadores municipais, mas não há, neste conjunto, desfecho SIA/SIH por município validado para calcular taxas municipais.


## Fonte

[IBGE — estimativas da população](https://www.ibge.gov.br/estatisticas/sociais/populacao/9103-estimativas-de-populacao.html), com referência em 1º de julho de cada ano.


## 1. Leitura e consistência dos registros

In [ ]:
from pathlib import Path
import csv

BASES=[Path("../Dados"),Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados"),Path("Dados")]
def ler(nome):
    p=next((b/nome for b in BASES if (b/nome).exists()),BASES[0]/nome)
    if not p.exists(): raise FileNotFoundError(p)
    with p.open(encoding="utf-8-sig",newline="") as f: return list(csv.DictReader(f,delimiter=";"))
municipios=ler("populacao_ibge_municipio_2024_2025.csv")
ufs=ler("populacao_ibge_uf_2024_2025.csv")
assert municipios and len(municipios)==len({(r["ano"],r["cod_municipio_ibge"]) for r in municipios}), "Registros município/ano duplicados ou vazios."
assert set(r["ano"] for r in municipios)=={"2024","2025"}
for r in municipios:
    r["populacao_estimada"]=int(r["populacao_estimada"])
    assert len(r["cod_municipio_ibge"])==7
    assert r["populacao_estimada"]>0
print("Registros municipais:",len(municipios))
print("Municípios únicos 2024:",len({r["cod_municipio_ibge"] for r in municipios if r["ano"]=="2024"}))
print("Municípios únicos 2025:",len({r["cod_municipio_ibge"] for r in municipios if r["ano"]=="2025"}))


## 2. Reconciliar município → UF → Brasil

In [ ]:
popuf={(r["ano"],r["cod_uf"]):int(r["populacao_estimada"]) for r in ufs}
por_uf={}
for r in municipios:
    chave=(r["ano"],r["cod_uf"])
    por_uf[chave]=por_uf.get(chave,0)+r["populacao_estimada"]
assert set(por_uf)==set(popuf), "Conjunto de UFs/anos difere entre arquivo municipal e arquivo UF."
divergencias=[(ano,cod,por_uf[(ano,cod)],popuf[(ano,cod)]) for ano,cod in popuf if por_uf[(ano,cod)]!=popuf[(ano,cod)]]
assert not divergencias, f"Somas municipais não conciliam com UF: {divergencias[:10]}"
for ano in ("2024","2025"):
    print(ano,"total nacional calculado:",sum(v for (a,_),v in popuf.items() if a==ano))
print("OK: soma dos municípios reconcilia exatamente com cada UF nos dois anos.")


## 3. Maiores municípios e mudanças 2024–2025

In [ ]:
por_codigo={}
for r in municipios: por_codigo.setdefault(r["cod_municipio_ibge"],{})[r["ano"]]=r
assert all(set(v)=={"2024","2025"} for v in por_codigo.values()), "Município sem um dos anos."
print("10 municípios mais populosos em 2025:")
for r in sorted((v["2025"] for v in por_codigo.values()),key=lambda x:x["populacao_estimada"],reverse=True)[:10]:
    print(f"{r['municipio']} / {r['sigla_uf']}: {r['populacao_estimada']:,}".replace(",","."))

variacao=[]
for cod,v in por_codigo.items():
    a,b=v["2024"]["populacao_estimada"],v["2025"]["populacao_estimada"]
    variacao.append((v["2025"]["municipio"],v["2025"]["sigla_uf"],100*(b-a)/a,a,b))
print("\nMaiores variações relativas (mostrar com cautela: base pequena amplifica percentuais):")
for nome,uf,pct,a,b in sorted(variacao,key=lambda x:x[2],reverse=True)[:10]:
    print(f"{nome}/{uf}: {pct:.1f}% ({a:,} → {b:,})".replace(",","."))


### Limites para uso analítico

- A população municipal pode servir como denominador somente se numeradores forem geograficamente compatíveis e corresponderem à mesma referência temporal.
- Não inferir acesso, necessidade clínica ou desempenho assistencial a partir da população isolada.
- A classificação capital/interior não está criada aqui; ela requer uma regra explícita e documentada.
- Mudanças percentuais altas em municípios pequenos podem resultar de bases pequenas ou de mudanças de estimativa; examine valores absolutos e a edição do IBGE.
